# Matrix Spike (MS) Drift Detection

## Purpose

This notebook investigates sustained changes in historical Matrix Spike
behaviour over time.

The analysis aims to:

- detect gradual movement away from the supplied MS target;
- analyse drift separately within Scheme–Analyte–Unit groups;
- distinguish sustained movement from isolated individual QC failures;
- identify the direction and approximate start of detected drift;
- provide candidate drift outputs for subsequent validation.

Existing CCLAS Warning and Failure classifications are retained for comparison.

This notebook detects candidate drift patterns. Formal validation of the
outputs is handled separately.

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [26]:
# Find workbook
matches = list(Path("..").rglob("QC Anomaly Training Data*.xlsx"))

if len(matches) == 0:
    raise FileNotFoundError("QC Anomaly Training Data workbook not found.")

data_file = matches[0]

print("Using file:", data_file)

ms = pd.read_excel(
    data_file,
    sheet_name="SPK(MS) Assessment"
)

ms = ms.dropna(axis=1, how="all")

print("Matrix Spike rows:", len(ms))

Using file: ../data/raw/QC Anomaly Training Data v2 (1).xlsx
Matrix Spike rows: 6958


In [27]:
drift_columns = [
    "SCHEME_CODE",
    "ANALYTE_CODE",
    "UNIT_CODE",
    "ANALYSED_DATE",
    "NUMERIC_FINAL_VALUE",
    "INTERNAL_TARGET_VALUE",
    "INTERNAL_MIN_WARNING_VALUE",
    "INTERNAL_MAX_WARNING_VALUE",
    "INTERNAL_MIN_VALUE",
    "INTERNAL_MAX_VALUE",
    "STANDARD_STATUS"
]

ms_drift = ms[drift_columns].copy()

ms_drift["ANALYSED_DATE"] = pd.to_datetime(
    ms_drift["ANALYSED_DATE"],
    errors="coerce"
)

ms_drift = ms_drift.dropna(
    subset=[
        "SCHEME_CODE",
        "ANALYTE_CODE",
        "UNIT_CODE",
        "ANALYSED_DATE",
        "NUMERIC_FINAL_VALUE",
        "INTERNAL_TARGET_VALUE",
        "INTERNAL_MIN_WARNING_VALUE",
        "INTERNAL_MAX_WARNING_VALUE"
    ]
)

ms_drift = ms_drift.sort_values(
    [
        "SCHEME_CODE",
        "ANALYTE_CODE",
        "UNIT_CODE",
        "ANALYSED_DATE"
    ]
)

print("Usable drift rows:", len(ms_drift))

Usable drift rows: 6872


In [28]:
ms_drift["DEVIATION"] = (
    ms_drift["NUMERIC_FINAL_VALUE"]
    - ms_drift["INTERNAL_TARGET_VALUE"]
)

## Normalised Deviation

Matrix Spike analytes operate on different numerical scales, so raw deviation
values are not directly comparable.

Deviation is normalised relative to the supplied warning boundaries:

- 0 represents the supplied target
- +1 represents the upper warning boundary
- -1 represents the lower warning boundary

This provides a consistent scale for the drift-detection logic.

In [29]:
upper_warning_distance = (
    ms_drift["INTERNAL_MAX_WARNING_VALUE"]
    - ms_drift["INTERNAL_TARGET_VALUE"]
)

lower_warning_distance = (
    ms_drift["INTERNAL_TARGET_VALUE"]
    - ms_drift["INTERNAL_MIN_WARNING_VALUE"]
)

# Prevent division by zero
upper_warning_distance = upper_warning_distance.replace(0, np.nan)
lower_warning_distance = lower_warning_distance.replace(0, np.nan)

ms_drift["NORMALISED_DEVIATION"] = np.where(
    ms_drift["DEVIATION"] >= 0,
    ms_drift["DEVIATION"] / upper_warning_distance,
    ms_drift["DEVIATION"] / lower_warning_distance
)

display(
    ms_drift[
        [
            "SCHEME_CODE",
            "ANALYTE_CODE",
            "UNIT_CODE",
            "ANALYSED_DATE",
            "DEVIATION",
            "NORMALISED_DEVIATION",
            "STANDARD_STATUS"
        ]
    ].head(10)
)

,SCHEME_CODE,ANALYTE_CODE,UNIT_CODE,ANALYSED_DATE,DEVIATION,NORMALISED_DEVIATION,STANDARD_STATUS
1363,GE_ICP40Q12,AG,MG_KG,2020-06-29 11:14:08,0.455450,0.540751,Pass
1361,GE_ICP40Q12,AG,MG_KG,2020-07-08 05:53:05,0.764689,0.907907,Pass
1360,GE_ICP40Q12,AG,MG_KG,2020-07-08 06:49:31,0.871000,1.034129,UpperWarning
1357,GE_ICP40Q12,AG,MG_KG,2020-07-09 12:36:17,0.496510,0.589501,Pass
1355,GE_ICP40Q12,AG,MG_KG,2020-07-10 03:22:44,0.882376,1.047635,UpperWarning
1352,GE_ICP40Q12,AG,MG_KG,2020-07-13 13:54:29,0.381000,0.452357,Pass
1351,GE_ICP40Q12,AG,MG_KG,2020-07-14 01:37:45,-0.059000,-0.070050,Pass
1350,GE_ICP40Q12,AG,MG_KG,2020-07-15 11:56:22,0.562176,0.667466,Pass
1349,GE_ICP40Q12,AG,MG_KG,2020-07-17 12:24:41,0.469731,0.557706,Pass
1348,GE_ICP40Q12,AG,MG_KG,2020-07-20 05:29:35,0.649571,0.771229,Pass


## Rolling Historical Behaviour

To detect sustained movement rather than isolated individual results,
a rolling mean is calculated for the normalised deviation within each
Scheme–Analyte–Unit group.

A rolling window smooths short-term fluctuations and shows whether several
consecutive Matrix Spike results are collectively moving above or below
their target.

At this stage, the rolling mean is descriptive and is not yet a final
drift classification.

In [30]:
ROLLING_WINDOW = 10

group_cols = [
    "SCHEME_CODE",
    "ANALYTE_CODE",
    "UNIT_CODE"
]

ms_drift["ROLLING_MEAN"] = (
    ms_drift
    .groupby(group_cols)["NORMALISED_DEVIATION"]
    .transform(
        lambda x: x.rolling(
            window=ROLLING_WINDOW,
            min_periods=ROLLING_WINDOW
        ).mean()
    )
)

display(
    ms_drift[
        [
            "SCHEME_CODE",
            "ANALYTE_CODE",
            "UNIT_CODE",
            "ANALYSED_DATE",
            "NORMALISED_DEVIATION",
            "ROLLING_MEAN",
            "STANDARD_STATUS"
        ]
    ].head(20)
)

,SCHEME_CODE,ANALYTE_CODE,UNIT_CODE,ANALYSED_DATE,NORMALISED_DEVIATION,ROLLING_MEAN,STANDARD_STATUS
1363,GE_ICP40Q12,AG,MG_KG,2020-06-29 11:14:08,0.540751,NaN,Pass
1361,GE_ICP40Q12,AG,MG_KG,2020-07-08 05:53:05,0.907907,NaN,Pass
1360,GE_ICP40Q12,AG,MG_KG,2020-07-08 06:49:31,1.034129,NaN,UpperWarning
1357,GE_ICP40Q12,AG,MG_KG,2020-07-09 12:36:17,0.589501,NaN,Pass
1355,GE_ICP40Q12,AG,MG_KG,2020-07-10 03:22:44,1.047635,NaN,UpperWarning
1352,GE_ICP40Q12,AG,MG_KG,2020-07-13 13:54:29,0.452357,NaN,Pass
1351,GE_ICP40Q12,AG,MG_KG,2020-07-14 01:37:45,-0.070050,NaN,Pass
1350,GE_ICP40Q12,AG,MG_KG,2020-07-15 11:56:22,0.667466,NaN,Pass
1349,GE_ICP40Q12,AG,MG_KG,2020-07-17 12:24:41,0.557706,NaN,Pass
1348,GE_ICP40Q12,AG,MG_KG,2020-07-20 05:29:35,0.771229,0.649863,Pass


## Candidate Drift Detection

Candidate drift is identified from sustained movement in the rolling mean of
normalised deviation.

For the initial prototype:

- rolling mean above +0.5 indicates potential upward movement;
- rolling mean below -0.5 indicates potential downward movement;
- the threshold must be exceeded for at least three consecutive observations
  before the sequence is treated as a drift candidate.

These are initial prototype parameters and require validation on the Matrix
Spike data.

In [31]:
DRIFT_THRESHOLD = 0.5
CONSECUTIVE_BREACHES = 3

ms_drift["DRIFT_DIRECTION"] = "None"

ms_drift.loc[
    ms_drift["ROLLING_MEAN"] > DRIFT_THRESHOLD,
    "DRIFT_DIRECTION"
] = "Upward"

ms_drift.loc[
    ms_drift["ROLLING_MEAN"] < -DRIFT_THRESHOLD,
    "DRIFT_DIRECTION"
] = "Downward"

In [32]:
def detect_consecutive_drift(group):
    group = group.copy()

    upward = group["ROLLING_MEAN"] > DRIFT_THRESHOLD
    downward = group["ROLLING_MEAN"] < -DRIFT_THRESHOLD

    upward_run = (
        upward.astype(int)
        .groupby((~upward).cumsum())
        .cumsum()
    )

    downward_run = (
        downward.astype(int)
        .groupby((~downward).cumsum())
        .cumsum()
    )

    group["DRIFT_CANDIDATE"] = (
        (upward_run >= CONSECUTIVE_BREACHES) |
        (downward_run >= CONSECUTIVE_BREACHES)
    )

    return group


ms_drift = (
    ms_drift
    .groupby(
        ["SCHEME_CODE", "ANALYTE_CODE", "UNIT_CODE"],
        group_keys=False
    )
    .apply(detect_consecutive_drift)
)

In [33]:
display(
    ms_drift[
        [
            "SCHEME_CODE",
            "ANALYTE_CODE",
            "UNIT_CODE",
            "ANALYSED_DATE",
            "NORMALISED_DEVIATION",
            "ROLLING_MEAN",
            "DRIFT_DIRECTION",
            "DRIFT_CANDIDATE",
            "STANDARD_STATUS"
        ]
    ].head(30)
)

KeyError: "['SCHEME_CODE', 'ANALYTE_CODE', 'UNIT_CODE'] not in index"

In [ ]:
DRIFT_THRESHOLD = 0.5
CONSECUTIVE_BREACHES = 3

# Identify whether each rolling mean is above or below the candidate drift threshold
ms_drift["DRIFT_DIRECTION"] = "None"

ms_drift.loc[
    ms_drift["ROLLING_MEAN"] > DRIFT_THRESHOLD,
    "DRIFT_DIRECTION"
] = "Upward"

ms_drift.loc[
    ms_drift["ROLLING_MEAN"] < -DRIFT_THRESHOLD,
    "DRIFT_DIRECTION"
] = "Downward"